# 09 - Predictive modeling: logistic baseline + XGBoost + RF

Out-of-sample prediction of `is_otp` (terminal lateness < 6 min) using zero information about the run
in progress - every feature is schedule/calendar/weather-as-of-departure/lagged-history only.

Train: `year <= 2024`. Test: `year == 2025`, restricted to `service_date <= 2025-08-26` (weather coverage ends there; seasonal robustness check at end of `09e` suggests there is not an issue with this cutoff/seasonality). Logistic regression is the interpretable baseline; XGBoost and RF are compared against it. XGBoost is the GBM used throughout the rest of the pipeline

## Setup

In [1]:
import time as _time

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
)
import xgboost as xgb

from utils import (
    load_model_split, prep_gbm_matrices, GBM_FEATURES, GBM_CAT_FEATURES,
)

BASEPATH = "../data"

# temporary progress instrumentation for long-running cells below -- tail
# /tmp/09_progress.log from a separate shell to check in on a background run
PROGRESS_LOG = "/tmp/09_progress.log"

def log_progress(msg):
    ts = _time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(PROGRESS_LOG, "a") as f:
        f.write(line + "\n")

log_progress("09_prediction.ipynb started")

train_df, test_df = load_model_split(BASEPATH)
print(f"train: {train_df.shape}, OTP rate {train_df['is_otp'].mean():.3f}")
print(f"test:  {test_df.shape}, OTP rate {test_df['is_otp'].mean():.3f}")

[08:45:05] 09_prediction.ipynb started


train: (1357563, 107), OTP rate 0.834
test:  (126332, 107), OTP rate 0.786


## Logistic regression (line fixed effects)
Compact predictor set for interpretability, not the full `GBM_FEATURES` list.

In [2]:
line_fe_formula = (
    "is_otp ~ "
    "C(line) + "
    "C(wx_primary_asof_collapsed, Treatment(reference='A_CLR')) + "
    "temp_c_asof + wind_speed_ms_asof + precip_mm_asof + "
    "any_gust_24h + any_ice_24h + has_cloud_layer + "
    "C(am_peak_overlap_bin) + "
    "C(pm_peak_x_sports, Treatment(reference='none_x_none')) + "
    "C(day_of_week) + is_holiday + sched_duration_sec + is_inbound + "
    "lag_line_lateness_min + lag_network_lateness_2hr_mean + "
    "lag_train_lateness_5run_mean"
)

_t0 = _time.time()
log_progress("Fitting logistic regression (line FE)...")
logit = smf.logit(line_fe_formula, data = train_df).fit()
log_progress(f"Logistic regression fit complete ({_time.time() - _t0:.1f}s)")
print(logit.summary())

[08:45:06] Fitting logistic regression (line FE)...


Optimization terminated successfully.
         Current function value: 0.396918
         Iterations 7


[08:45:21] Logistic regression fit complete (14.8s)


                           Logit Regression Results                           
Dep. Variable:                 is_otp   No. Observations:              1351482
Model:                          Logit   Df Residuals:                  1351410
Method:                           MLE   Df Model:                           71
Date:                Fri, 07 Aug 2026   Pseudo R-squ.:                  0.1174
Time:                        08:45:22   Log-Likelihood:            -5.3643e+05
converged:                       True   LL-Null:                   -6.0777e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------------------------------------------
Intercept                                                                 

## XGBoost (native categorical splitting)

In [3]:
X_train_gbm, X_test_gbm = prep_gbm_matrices(train_df, test_df)
y_train = train_df["is_otp"]
y_test = test_df["is_otp"]

_t0 = _time.time()
log_progress("Fitting XGBoost (default settings)...")
gbm = xgb.XGBClassifier(
    random_state = 42, enable_categorical = True, tree_method = "hist"
)
gbm.fit(X_train_gbm, y_train)
log_progress(f"XGBoost fit complete ({_time.time() - _t0:.1f}s)")

[08:45:22] Fitting XGBoost (default settings)...


[08:45:29] XGBoost fit complete (6.7s)


## Random forest (one-hot encoded)

RF sees a one-hot representation of the
same underlying categorical features. Kept
small for a quick comparison

In [4]:
X_train_rf = pd.get_dummies(
    train_df[GBM_FEATURES], columns = GBM_CAT_FEATURES
)
X_test_rf = pd.get_dummies(test_df[GBM_FEATURES], columns = GBM_CAT_FEATURES)
X_test_rf = X_test_rf.reindex(columns = X_train_rf.columns, fill_value = 0)
print(
    f"One-hot feature count: {X_train_rf.shape[1]} "
    f"(from {len(GBM_FEATURES)} raw features)"
)

_t0 = _time.time()
log_progress(
    f"Fitting RandomForestClassifier (n_estimators=200, "
    f"{len(train_df):,} rows, {X_train_rf.shape[1]} one-hot columns) "
)
rf = RandomForestClassifier(
    n_estimators = 200, max_depth = 20, min_samples_leaf = 20,
    n_jobs = -1, random_state = 42,
)
rf.fit(X_train_rf, y_train)
log_progress(f"RF fit complete ({_time.time() - _t0:.1f}s)")

One-hot feature count: 134 (from 71 raw features)
[08:45:29] Fitting RandomForestClassifier (n_estimators=200, 1,357,563 rows, 134 one-hot columns) 


[08:47:18] RF fit complete (109.4s)


## Evaluation

Logit is restricted to its own complete-case subset (it can't score rows missing any of its predictors);
XGBoost and random forest handle missing values natively, so they're evaluated on the full test set
instead of being artificially restricted to match -- the gap is small (a few hundred rows, well under 1%
of the test set).

In [5]:
logit_predictor_cols = [
    "line", "wx_primary_asof_collapsed", "temp_c_asof",
    "wind_speed_ms_asof", "precip_mm_asof", "any_gust_24h",
    "any_ice_24h", "has_cloud_layer", "am_peak_overlap_bin",
    "pm_peak_x_sports", "day_of_week", "is_holiday",
    "sched_duration_sec", "is_inbound", "lag_line_lateness_min",
    "lag_network_lateness_2hr_mean", "lag_train_lateness_5run_mean",
]
complete_mask = test_df[logit_predictor_cols].notna().all(axis = 1)
print(
    f"Test rows excluded (logit complete-case): "
    f"{(~complete_mask).sum():,} / {len(test_df):,} "
    f"({(~complete_mask).mean():.2%})"
)

test_df_cc = test_df[complete_mask]
y_test_cc = y_test[complete_mask]

p_logit = logit.predict(test_df_cc)
p_gbm = gbm.predict_proba(X_test_gbm)[:, 1]
p_rf = rf.predict_proba(X_test_rf)[:, 1]
p_naive = np.full(len(y_test), train_df["is_otp"].mean())

Test rows excluded (logit complete-case): 496 / 126,332 (0.39%)


In [6]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, balanced_accuracy_score,
)

def evaluate(name, y_true, p_otp, threshold = 0.5):
    y_late = 1 - y_true
    p_late = 1 - p_otp
    pred = (p_otp >= threshold).astype(int)
    pred_late = 1 - pred
    return {
        "model": name,
        "accuracy": (pred == y_true).mean(),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, p_otp),
        "pr_auc_late": average_precision_score(y_late, p_late),
        "precision_late": precision_score(y_late, pred_late, zero_division = 0),
        "recall_late": recall_score(y_late, pred_late, zero_division = 0),
        "f1_late": f1_score(y_late, pred_late, zero_division = 0),
        "brier": brier_score_loss(y_true, p_otp),
    }

results = pd.DataFrame([
    evaluate("naive (train base rate)", y_test, p_naive),
    evaluate("logit: line FE", y_test_cc, p_logit),
    evaluate("XGBoost", y_test, p_gbm),
    evaluate("random forest", y_test, p_rf),
]).set_index("model")

log_progress("Baseline model evaluation complete")
print(results.to_string())

[08:47:20] Baseline model evaluation complete
                         accuracy  balanced_accuracy   roc_auc  pr_auc_late  precision_late  recall_late   f1_late     brier
model                                                                                                                       
naive (train base rate)  0.786317           0.500000  0.500000     0.213683        0.000000     0.000000  0.000000  0.170304
logit: line FE           0.799731           0.578607  0.732195     0.455117        0.594399     0.192915  0.291291  0.147553
XGBoost                  0.832394           0.668016  0.807156     0.613955        0.697362     0.380959  0.492741  0.123637
random forest            0.828792           0.639544  0.793102     0.588565        0.737014     0.309057  0.435495  0.128936


### Cache baseline predictions

Caches GBM/RF predictions on the full test set -- `09b_tuning.ipynb`'s go/no-go baseline reuses this
directly instead of refitting the identical models.

In [7]:
pd.DataFrame({
    "is_otp": y_test.values,
    "p_gbm": p_gbm,
    "p_rf": p_rf,
}, index = test_df.index).to_parquet(
    f"{BASEPATH}/9_baseline_predictions_test.parquet"
)
log_progress(
    "Cached baseline predictions to 9_baseline_predictions_test.parquet"
)

[08:47:20] Cached baseline predictions to 9_baseline_predictions_test.parquet
